# Data Scraping: API Calls for Energy Prices, Weather Actuals and Weather Forecasts

All raw data is fetched from two public APIs:

| Source | API | Data |
|--------|-----|------|
| Energi Data Service | `api.energidataservice.dk` | Day-ahead electricity prices, hourly consumption |
| Open-Meteo (archive) | `archive-api.open-meteo.com` | Historical weather actuals (2021–present) |
| Open-Meteo (forecasts) | `previous-runs-api.open-meteo.com` | NWP previous-run forecasts (2025–present) |

### Day-ahead electricity prices
Fetched from two overlapping Energi Data Service datasets — `Elspotprices` (older history) and `DayAheadPrices` (more recent). Both are normalised to the same schema, merged, and deduplicated. The result contains one row per hour with prices in EUR/MWh and DKK/MWh. DK1 covers western Denmark, where prices are strongly driven by wind power availability, interconnection flows, and demand variation.

### Hourly electricity consumption
Also from Energi Data Service. Returns consumption at grid-area level, which is aggregated to DK1 as a whole. Used as a proxy for grid stress — hours with high system-level demand are treated as peak-load periods in the flexibility simulation.

### Weather actuals
Eight hourly variables fetched from the Open-Meteo archive for DK1 West (lat 56.15, lon 8.45) using the ECMWF IFS model:

| Variable | Relevance |
|----------|-----------|
| Wind speed & direction (10 m, 100 m) | Wind turbine production proxy; 100 m is closer to hub height |
| Shortwave radiation | Solar PV production proxy |
| Cloud cover | Affects solar production |
| Temperature 2 m | Heating and cooling demand |
| Pressure MSL | Large-scale weather regime signal |

These actuals are used both to estimate NWP forecast error distributions and as the base for generating synthetic forecasts for years before 2025.

### NWP weather forecasts (Previous Runs API)
The Open-Meteo Previous Runs API stores the *as-issued* ECMWF forecast for each day, going back roughly 12 months. For each target hour it returns the forecast as it looked 1, 2, 3, 4, and 5 days before (columns `_previous_day1` through `_previous_day5`). These are real forecast values — not reanalysis — making them the gold-standard source for measuring how large NWP errors are at each lead time. Data is only available from January 2025 onwards; for earlier years, synthetic forecasts are generated in the processing step.

---
## Fetching all data

`fetch_all()` runs all four sources in sequence and saves them to the `data/` folder.

In [ ]:
from src.data.data_collection import fetch_all

results = fetch_all(start="2021-01-01", end="2026-04-28", price_area="DK1")

This produces four files:

| File | Content |
|------|---------|
| `data/weather_actuals_raw.csv` | Hourly weather observations 2021–present |
| `data/weather_forecasts_raw.csv` | NWP previous-run forecasts 2025–present |
| `data/consumption_dk1_raw.csv` | Hourly DK1 consumption 2021–present |
| `data/day_ahead_prices_dk1_raw.csv` | Hourly DK1 day-ahead prices 2021–present |

---
## Next step: Data processing

The raw files feed into `src/data/data_processing.py`, which:
1. Estimates NWP error distributions at five lead times (24–120 h) → `data/weather_error_distributions.csv`
2. Simulates 120-hour forecasts for the full 2021–present period → `data/forecast_dataset.parquet`

```python
from src.data.data_processing import build_error_distributions, build_forecast_dataset

build_error_distributions()
build_forecast_dataset()
```